In [ ]:
import json
import re
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from groq import Groq

load_dotenv(r"C:\Users\onmyb\Desktop\bootcamp-da-p2\Proyecto_7\notebooks\.env", override=True)

client_groq = Groq(api_key=os.getenv("GROQ_API_KEY"))

print(os.getenv("GROQ_API_KEY")[:10])  # verificar que carga bien

In [ ]:


print(Path("../.env").exists())

In [ ]:

print(Path.cwd())
print(Path("../.env"))
print(Path("../.env").exists())

In [ ]:
from pathlib import Path

for f in Path("C:/Users/onmyb/Desktop/bootcamp-da-p2").rglob(".env"):
    print(f)

In [ ]:
load_dotenv(
    r"C:\Users\onmyb\Desktop\bootcamp-da-p2\Proyecto_7\notebooks\.env",
    override=True
)

In [ ]:
key = os.getenv("GOOGLE_API_KEY")

print(key[:10] if key else "NO KEY")

In [ ]:
import pandas as pd

df = pd.read_csv("../data/processed/master_fra.csv")

print(df.shape)

In [ ]:
df["category"].value_counts()

In [ ]:
questions = (
    df[["category", "question_code", "question_label"]]
    .drop_duplicates()
    .sort_values(["category", "question_code"])
)

In [ ]:
pd.set_option("display.max_colwidth", None)

questions.head(20)

In [ ]:
df[
    df["question_code"] == "b1_a"
][["answer", "percentage"]].head(20)

In [ ]:
questions.to_excel(
    "../data/processed/question_inventory.xlsx",
    index=False
)

In [ ]:
questions["block"] = questions["question_code"].str.extract(r"([a-z]+\d+)")

questions["block"].value_counts()

Tabla resumen de bloques

In [ ]:
blocks = (
    questions
    .groupby("block")
    .agg({
        "question_label": "first"
    })
    .reset_index()
)

blocks.shape

In [ ]:
pd.set_option("display.max_colwidth", None)

blocks.head(20)

In [ ]:
blocks.to_excel(
    "../data/processed/block_inventory.xlsx",
    index=False
)

print("Archivo generado correctamente")

In [ ]:
block_classification = blocks.copy()

block_classification["relevant"] = ""
block_classification["direction"] = ""
block_classification["theme"] = ""

block_classification.head()

In [ ]:
import json
import time

In [ ]:
def classify_block(question_label):

    prompt = f"""
    You are helping to build an LGBT Acceptance Index.

    Analyze this survey question:

    {question_label}

    Return ONLY valid JSON.

    {{
        "relevant": "yes" or "no",
        "direction": "positive", "negative" or "neutral",
        "theme": "short thematic category"
    }}
    """

    response = client.models.generate_content(
        model="gemini-2.0-flash-lite",
        contents=prompt
    )

    print("========== GEMINI RESPONSE ==========")
    print(response.text)
    print("====================================")

    raw = response.text.strip()
    raw = re.sub(r"```json|```", "", raw).strip()  # elimina backticks si Gemini los añade
    return json.loads(raw)

In [ ]:
import json
import re

# Preparar lista numerada de bloques
blocks_list = "\n".join([
    f"{i+1}. block='{row['block']}' | question='{row['question_label'][:150]}'"
    for i, (_, row) in enumerate(blocks.iterrows())
])

prompt = f"""
You are helping build an LGBT Acceptance Index from EU survey data.

Classify each of the following {len(blocks)} survey question blocks.

For each block determine:
- relevant: "yes" if it measures LGBTI attitudes, experiences or rights. "no" if purely demographic (age, citizenship, etc.)
- direction: "positive" if high % = more acceptance, "negative" if high % = less acceptance, "neutral" if it doesn't map cleanly
- theme: one short label e.g. "workplace_discrimination", "legal_rights", "social_comfort", "political_hostility", "healthcare", "school_safety"

BLOCKS:
{blocks_list}

Return ONLY a valid JSON array with exactly {len(blocks)} objects in the same order:
[
  {{"block": "a11", "relevant": "no", "direction": "neutral", "theme": "demographics"}},
  ...
]
No markdown, no explanation, just the JSON array.
"""

response = client_groq.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}]
)

raw = response.choices[0].message.content.strip()
raw = re.sub(r"```json|```", "", raw).strip()
classifications = json.loads(raw)

df_class = pd.DataFrame(classifications)
block_classification = blocks.merge(df_class, on="block", how="left")

print(f"✅ Clasificados {len(block_classification)} bloques")
block_classification.head()

In [ ]:
# Correcciones manuales sobre la clasificación de la IA

corrections = {
    # coming out personal, no mide aceptación social
    "a13": {"relevant": "no", "direction": "neutral", "theme": "demographics"},
    "a14": {"relevant": "no", "direction": "neutral", "theme": "demographics"},
    
    # localización de incidentes, no tiene porcentaje útil para el índice
    "fa1": {"relevant": "no", "direction": "neutral", "theme": "violence_location"},
    "fa2": {"relevant": "no", "direction": "neutral", "theme": "violence_location"},
    "fb1": {"relevant": "no", "direction": "neutral", "theme": "harassment_location"},
    "fb2": {"relevant": "no", "direction": "neutral", "theme": "harassment_location"},
    
    # apertura personal, no mide aceptación social
    "g1": {"relevant": "yes", "direction": "neutral", "theme": "social_openness"},
    "g2": {"relevant": "yes", "direction": "neutral", "theme": "social_openness"},
    "g3": {"relevant": "yes", "direction": "neutral", "theme": "social_openness"},
    
    # bienestar LGBTI, sí es relevante
    "h19": {"relevant": "yes", "direction": "positive", "theme": "wellbeing"},
}

for block, fields in corrections.items():
    for col, val in fields.items():
        block_classification.loc[block_classification["block"] == block, col] = val

print(f"✅ Correcciones aplicadas: {len(corrections)} bloques ajustados")

# Sobreescribir el Excel con la versión corregida
block_classification.to_excel("../data/processed/block_classification_ai.xlsx", index=False)
print("✅ Excel actualizado con correcciones")

In [ ]:
# Convertir percentage a numérico
df_filtered["percentage"] = pd.to_numeric(df_filtered["percentage"], errors="coerce")

In [ ]:
print(df_filtered["percentage"].dtype)
print(df_filtered["percentage"].head(10))

In [55]:
# ============================================================
# CÁLCULO DEL ÍNDICE DE ACEPTACIÓN POR PAÍS Y AÑO
# ============================================================

# 1. Mapa de pesos base (asume dirección positiva)
answer_weights = {
    # Escala widespread
    "Very widespread":    0.0,
    "Fairly widespread":  0.33,
    "Fairly rare":        0.66,
    "Very rare":          1.0,

    # Escala frecuencia
    "Always":   1.0,
    "Often":    0.66,
    "Rarely":   0.33,
    "Never":    0.0,

    # Escala acuerdo
    "Strongly agree":    1.0,
    "Agree":             0.66,
    "Disagree":          0.33,
    "Strongly disagree": 0.0,

    # Escala cantidad
    "All":    1.0,
    "Most":   0.66,
    "A few":  0.33,

    # Escala apertura
    "Very open": 1.0,

    # Escala cambio
    "Increased a lot":    1.0,
    "Increased a little": 0.66,
    "Stayed the same":    0.5,
    "Decreased a little": 0.33,
    "Decreased a lot":    0.0,

    # Binarias
    "Yes": 1.0,
    "No":  0.0,
}

# Respuestas a excluir del cálculo
exclude_answers = {
    "Don`t know", "Dont know", "Do not know",
    "Other", "Prefer not to say", "None of the above",
    "Current situation is fine"
}

# 2. Añadir columna block al dataframe principal
df["block"] = df["question_code"].str.extract(r"([a-z]+\d+)")

df["percentage"] = pd.to_numeric(df["percentage"], errors="coerce")

# 3. Merge con la clasificación
df_merged = df.merge(
    block_classification[["block", "relevant", "direction"]],
    on="block",
    how="left"
)

# 4. Filtrar solo bloques relevantes y respuestas válidas
df_filtered = df_merged[
    (df_merged["relevant"] == "yes") &
    (~df_merged["answer"].isin(exclude_answers)) &
    (df_merged["answer"].isin(answer_weights.keys()))
].copy()

# 5. Asignar peso base según respuesta
df_filtered["weight"] = df_filtered["answer"].map(answer_weights)

# 6. Invertir peso si la dirección es negativa
# (ej: "Very widespread" en bloque negativo = mala señal = peso bajo)
df_filtered.loc[df_filtered["direction"] == "negative", "weight"] = (
    1 - df_filtered.loc[df_filtered["direction"] == "negative", "weight"]
)

# 7. Calcular score ponderado por respuesta
# score = weight * (percentage / 100)
df_filtered["score"] = df_filtered["weight"] * (df_filtered["percentage"] / 100)

# 8. Agregar por país y año → índice de aceptación (0-100)
acceptance_index = (
    df_filtered
    .groupby(["year", "CountryCode"])
    .agg(
        score_mean=("score", "mean"),
        n_responses=("score", "count")
    )
    .reset_index()
)

acceptance_index["acceptance_index"] = (acceptance_index["score_mean"] * 100).round(2)
acceptance_index = acceptance_index.drop(columns="score_mean")

print(f"✅ Índice calculado: {len(acceptance_index)} combinaciones país-año")
print(acceptance_index.head(20))

# 9. Exportar
acceptance_index.to_excel("../data/processed/acceptance_index.xlsx", index=False)
acceptance_index.to_csv("../data/processed/acceptance_index.csv", index=False)
print("✅ Exportado a Excel y CSV")

✅ Índice calculado: 60 combinaciones país-año
    year     CountryCode  n_responses  acceptance_index
0   2012         Austria         1203             18.68
1   2012         Average         1213             18.44
2   2012         Belgium         1203             18.71
3   2012        Bulgaria         1197             17.29
4   2012         Croatia         1197             18.21
5   2012          Cyprus          599             16.57
6   2012  Czech Republic         1201             17.60
7   2012         Denmark         1201             19.06
8   2012         Estonia         1020             17.37
9   2012         Finland         1203             18.98
10  2012          France         1203             18.80
11  2012         Germany         1209             18.50
12  2012          Greece         1197             17.15
13  2012         Hungary         1201             17.65
14  2012         Ireland         1201             18.79
15  2012           Italy         1199             18.15
16

In [ ]:
output_path = "../data/processed/block_classification_ai.xlsx"

block_classification.to_excel(output_path, index=False)

print(f"✅ Excel guardado en: {output_path}")

In [ ]:
print(block_classification.shape)
print(block_classification.columns.tolist())

In [ ]:
print(df.shape)
print(df.columns.tolist())
df.head(3)

In [ ]:
print(df["answer"].value_counts().head(30))

In [ ]:
print(df["answer"].unique())

In [ ]:
# Filtrar solo bloques relevantes
relevant_blocks = block_classification[block_classification["relevant"] == "yes"]["block"].tolist()

# Extraer el prefijo del block de question_code
df["block"] = df["question_code"].str.extract(r"([a-z]+\d+)")

# Ver qué respuestas aparecen en bloques relevantes
df_relevant = df[df["block"].isin(relevant_blocks)]

print(df_relevant["answer"].value_counts().head(50))

In [58]:
import wbgapi as wb
from functools import reduce

# Años disponibles en vuestro dataset
years = sorted(df["year"].unique().tolist())
print(f"Años detectados en el dataset: {years}")

# Indicadores del Banco Mundial
indicators = {
    "NY.GDP.PCAP.CD":    "gdp_per_capita",
    "SI.POV.GINI":       "gini_index",
    "SE.XPD.TOTL.GD.ZS": "education_spending",
    "SP.URB.TOTL.IN.ZS": "urbanization_rate",
    "SL.UEM.TOTL.ZS":    "unemployment_rate",
}

dfs = []

for code, name in indicators.items():
    try:
        raw = wb.data.DataFrame(code, time=years, labels=True)
        raw = raw.reset_index()  # economy pasa a columna

        # Convertir columnas YR2012, YR2019 a filas
        year_cols = [c for c in raw.columns if c.startswith("YR")]
        raw = raw.melt(
            id_vars=["economy", "Country"],
            value_vars=year_cols,
            var_name="year",
            value_name=name
        )

        # Limpiar año: "YR2012" → 2012
        raw["year"] = raw["year"].str.replace("YR", "").astype(int)
        raw = raw.rename(columns={"Country": "CountryName", "economy": "CountryCode_wb"})
        raw = raw[["CountryName", "year", name]]

        dfs.append(raw)
        print(f"✅ {name} descargado ({len(raw)} filas)")

    except Exception as e:
        print(f"⚠️ Error descargando {name}: {e}")

# Merge de todos los indicadores
wb_data = reduce(
    lambda left, right: left.merge(right, on=["CountryName", "year"], how="outer"),
    dfs
)

print(f"\n✅ World Bank data: {wb_data.shape}")
print(wb_data.head())

# Exportar
wb_data.to_csv("../data/processed/worldbank_indicators.csv", index=False)
print("✅ Exportado a worldbank_indicators.csv")

Años detectados en el dataset: [2012, 2019]
✅ gdp_per_capita descargado (532 filas)
✅ gini_index descargado (532 filas)
✅ education_spending descargado (532 filas)
✅ urbanization_rate descargado (532 filas)
✅ unemployment_rate descargado (532 filas)

✅ World Bank data: (532, 7)
                   CountryName  year  gdp_per_capita  gini_index  \
0                  Afghanistan  2012      651.417134         NaN   
1                  Afghanistan  2019      496.602504         NaN   
2  Africa Eastern and Southern  2012     1732.038021         NaN   
3  Africa Eastern and Southern  2019     1507.085600         NaN   
4   Africa Western and Central  2012     1941.118243         NaN   

   education_spending  urbanization_rate  unemployment_rate  
0            2.604210          23.343439           7.875000  
1                 NaN          25.143726          11.187000  
2            4.676125          33.286706           7.108737  
3            4.511410          36.097331           7.459106  
4 

In [59]:
paises_fra = set(acceptance_index["CountryCode"].unique())
paises_wb = set(wb_data["CountryName"].unique())

print("Países en FRA:", len(paises_fra))
print("Países en WB:", len(paises_wb))
print("\nCoinciden exactamente:", len(paises_fra & paises_wb))
print("\nNo coinciden en FRA:", paises_fra - paises_wb)

Países en FRA: 33
Países en WB: 266

Coinciden exactamente: 29

No coinciden en FRA: {'Slovakia', 'Czech Republic', 'EU-28', 'Average'}


In [60]:
# Mapeo de nombres que no coinciden
name_mapping = {
    "Czech Republic": "Czechia",
    "Slovakia":       "Slovak Republic",
}

# Aplicar mapeo al índice de aceptación
acceptance_index["CountryName"] = acceptance_index["CountryCode"].replace(name_mapping)
acceptance_index["CountryName"] = acceptance_index["CountryName"].where(
    acceptance_index["CountryName"] != acceptance_index["CountryCode"],
    acceptance_index["CountryCode"]
)

# Eliminar filas que no son países reales
acceptance_index = acceptance_index[
    ~acceptance_index["CountryCode"].isin(["EU-28", "Average"])
]

# Merge con World Bank
df_model = acceptance_index.merge(
    wb_data,
    on=["CountryName", "year"],
    how="left"
)

print(f"✅ Dataset final: {df_model.shape}")
print(f"NaN por columna:\n{df_model.isnull().sum()}")
print(df_model.head())

# Exportar
df_model.to_csv("../data/processed/dataset_regresion.csv", index=False)
print("✅ Exportado a dataset_regresion.csv")

✅ Dataset final: (58, 10)
NaN por columna:
year                  0
CountryCode           0
n_responses           0
acceptance_index      0
CountryName           0
gdp_per_capita        0
gini_index            1
education_spending    6
urbanization_rate     0
unemployment_rate     0
dtype: int64
   year CountryCode  n_responses  acceptance_index CountryName  \
0  2012     Austria         1203             18.68     Austria   
1  2012     Belgium         1203             18.71     Belgium   
2  2012    Bulgaria         1197             17.29    Bulgaria   
3  2012     Croatia         1197             18.21     Croatia   
4  2012      Cyprus          599             16.57      Cyprus   

   gdp_per_capita  gini_index  education_spending  urbanization_rate  \
0    48250.405914        30.5             5.51613          67.208591   
1    44874.170918        27.5             6.56626          85.491540   
2     7431.957895        36.0             3.47650          72.768486   
3    13507.780526  

In [61]:
# Opción A: imputar con la media del país para gini_index y education_spending

for col in ["gini_index", "education_spending"]:
    # Media por país
    country_mean = df_model.groupby("CountryCode")[col].transform("mean")
    # Rellenar NaN con la media del país
    df_model[col] = df_model[col].fillna(country_mean)
    # Si sigue habiendo NaN (país sin ningún dato), rellenar con media general
    df_model[col] = df_model[col].fillna(df_model[col].mean())

print(f"NaN restantes:\n{df_model.isnull().sum()}")

# Sobreescribir el CSV con el dataset limpio
df_model.to_csv("../data/processed/dataset_regresion.csv", index=False)
print("✅ Dataset final limpio exportado")

NaN restantes:
year                  0
CountryCode           0
n_responses           0
acceptance_index      0
CountryName           0
gdp_per_capita        0
gini_index            0
education_spending    0
urbanization_rate     0
unemployment_rate     0
dtype: int64
✅ Dataset final limpio exportado
